**EEG–Eye-Tracking Multimodal Research | Presented by: Reza Saadatyar (2026)**

**02 | EEG–Gaze Alignment and Quality Assessment**

![EEG–Gaze Alignment and Quality Assessment](../../images/02_eeg_gaze_alignment_and_quality.svg)

**Objective:** establish a shared temporal window registry using real
       EEG and eye-stream BIDS metadata, sampling rates and recorded events.
       **Input:** Stage-01 source manifest and fixed participant roles.
       **Outputs:** paired-window registry, per-run quality table and aligned
       time-series example. A shared nominal 500-Hz sampling rate does **not**
       independently prove zero clock drift or physiologically exact timing.

**00. Colab Repository Setup**

In [ ]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess

repo_url = "https://github.com/RezaSaadatyar/Eye-Tracking-Neurocognitive-Intelligence.git"
repo_path = Path("/content") / repo_url.rsplit("/", 1)[-1].removesuffix(".git")
if not repo_path.exists():
    subprocess.run(
        ["git", "clone", "-q", repo_url, str(repo_path)], check=True
    )

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}


**01. Imports and Research Environment**

In [ ]:
# =================== 01. Imports and Research Environment ====================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("eeg_eye_tracking_fusion")

# ---------------------------- 2. Standard Library ----------------------------
import importlib.util
import json
import random
import shutil
import subprocess
import sys
from pathlib import Path

# ---------------------- 3. Colab Dependency Preparation ----------------------
if importlib.util.find_spec("mne") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "-q", "install",
         "mne>=1.6,<2"],
        check=True,
    )

# ----------------- 4. Scientific Computing and Data Analysis -----------------
import numpy as np
import pandas as pd

# ------------------- 5. Visualisation and Notebook Display -------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style

# ----------------- 6. Shared and Project Research Utilities ------------------
from core.output_download import download_outputs_from_colab
from zuco_io import open_run, session
from alignment import (
    window_registry, ensure_disjoint_registry, eye_sample_times,
    temporal_overlap,
)

# ---------------------------- 7. Reproducibility -----------------------------
SEED = 17
random.seed(SEED)
np.random.seed(SEED)

# ------------------- 8. Output Directory and Setup Summary -------------------
work = root / "eeg_eye_tracking_fusion"
output_dir = work / "output"
output_dir.mkdir(parents=True, exist_ok=True)
raw_dir = Path("/content/zuco2_source_cache")
raw_dir.mkdir(parents=True, exist_ok=True)
plot_style(font_family="DejaVu Sans", figure_dpi=100)
print("Project outputs:", output_dir)
print("Temporary source cache:", raw_dir)
print("Python:", sys.version.split()[0])
print("Free disk (GiB):", round(
    shutil.disk_usage("/content").free / 2**30, 2,
))


**02. Load Fixed Source Records and Alignment Settings**

Use precisely the original runs selected in Notebook 01.
          No missing or earlier outputs are substituted automatically.

In [ ]:
# =========== 02. Load Fixed Source Records and Alignment Settings ============


required = ['01_source_manifest.csv', '01_split_assignments.csv',
            '01_protocol.json']
for file in required:
    if not (output_dir / file).is_file():
        raise FileNotFoundError(f'Run Notebook 01 first: {file}')
source = pd.read_csv(output_dir / '01_source_manifest.csv',
                     dtype={'participant_id': str})
protocol = json.loads((output_dir / '01_protocol.json').read_text())
duration = float(protocol['window_seconds'])
max_windows = int(protocol['max_windows_per_run'])
min_eye_fraction = float(protocol['min_eye_fraction'])
REQUEST = session()
display(source[['participant_id', 'task', 'run', 'split']].head(10))


**03. Inspect Raw Temporal Overlap and Event Onsets**

Each selected BrainVision recording is read lazily. Gaze
          sample times are constructed using its own `StartTime` and sample
          frequency, relative to EEG recording start. Reading-event onsets
          must fall within the acquired record.

In [ ]:
# ============= 03. Inspect Raw Temporal Overlap and Event Onsets =============

run = source.iloc[0]
raw, eye, events, eye_meta, eeg_meta = open_run(run, raw_dir, REQUEST)
t_eye = eye_sample_times(eye_meta, len(eye))
overlap = temporal_overlap(raw.info['sfreq'], raw.n_times,
                           eye_meta, len(eye))
print('Selected run:', run.participant_id, run.task, run.run)
print('EEG samples:', raw.n_times, 'Eye samples:', len(eye))
print('Common seconds:', overlap, 'Eye start:', eye_meta['StartTime'])
print('Recorded event types:', events.trial_type.value_counts().head())
raw.close()


**04. Build Eligible Paired Windows for Every Run**

Default non-overlapping 12-s windows minimise near-duplicate
          observations; each covers one source task condition. Windows with
          insufficient finite eye position are marked ineligible. Keep the
          full registry to make exclusions auditable.

In [ ]:
# ============== 04. Build Eligible Paired Windows for Every Run ==============

window_parts, quality = [], []
for _, run in source.iterrows():
    raw, eye, events, eye_meta, _ = open_run(run, raw_dir, REQUEST)
    try:
        table, report = window_registry(
            run.to_dict(), raw, eye, eye_meta, events,
            window_seconds=duration,
            max_windows=max_windows,
            min_eye_fraction=min_eye_fraction)
        window_parts.append(table)
        quality.append(report)
    finally:
        raw.close()
    print(run.participant_id, run.task,
          report['eligible_windows'], 'eligible windows')
registry = pd.concat(window_parts, ignore_index=True)
report_table = pd.DataFrame(quality)
eligible = registry.loc[registry.eligible].copy()
ensure_disjoint_registry(registry)
if eligible.groupby(['participant_id', 'task']).size().unstack(
    fill_value=0).eq(0).any().any():
    raise ValueError('Some participant/task groups have no windows')
display(report_table[['participant_id', 'task',
                      'eligible_windows', 'eye_start_seconds']])


**05. Examine Eye Coverage and Source Event Alignment**

This quality plot shows the sampled eye-stream valid fraction
          by task and split. The BIDS metadata yield an alignment mapping,
          but independent timing-marker drift is not validated here.

In [ ]:
# ============ 05. Examine Eye Coverage and Source Event Alignment ============

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for task, part in registry.groupby('task'):
    axes[0].hist(part.eye_valid_fraction, bins=np.linspace(0, 1, 16),
                 alpha=0.6, label=task)
axes[0].axvline(min_eye_fraction, ls='--', color='black',
                label='Eligibility threshold')
axes[0].set(title='Eye-coordinate coverage per temporal window',
            xlabel='Finite gaze fraction', ylabel='Windows')
axes[0].legend()
report_table.groupby('task')[['eligible_windows']].mean().plot(
    kind='bar', ax=axes[1], legend=False)
axes[1].set(title='Eligible windows per original run',
            ylabel='Mean eligible windows', xlabel='Task')
fig.tight_layout()
fig.savefig(output_dir / '02_alignment_quality.png', dpi=170)
plt.show()


**06. Save Window Registry and Source Alignment Evidence**

Only the stage-owned registry and quality evidence are saved;
          no raw source signal is committed. Every downstream prediction
          refers to one `window_id`, split role and original participant.

In [ ]:
# ========== 06. Save Window Registry and Source Alignment Evidence ===========

registry.to_csv(output_dir / '02_window_registry.csv', index=False)
report_table.to_csv(output_dir / '02_alignment_quality.csv',
                    index=False)
summary = (registry.groupby(['split', 'task'])
           .agg(total=('window_id', 'size'),
                eligible=('eligible', 'sum'),
                people=('participant_id', 'nunique'))
           .reset_index())
summary.to_csv(output_dir / '02_window_summary.csv', index=False)
display(summary)
assert registry.window_id.is_unique
assert registry.groupby('participant_id').split.nunique().eq(1).all()


**07. Interpretation and Next Dependency**

The eligible paired windows support feature extraction, not
          a claim of independently validated signal-clock precision.
          Notebook 03 will derive modality-specific features only from
          these fixed windows and will retain their source identities.

In [ ]:
# ================== 07. Interpretation and Next Dependency ===================

print('Eligible windows:', int(eligible.shape[0]))
print('Independent people:', eligible.participant_id.nunique())
print('Independent drift checked:', False)
print('Next: 03_preprocessing_features_and_neurophysiology.ipynb')

# ---------------------- 2. Package Stage-Owned Outputs -----------------------
download_outputs_from_colab(
    "eeg_eye_tracking_fusion/output", "02"
)
